# Nettoyage des données du Congrès (2014-2026) — table de recherche CANONIQUE

Entonnoir **A→D** sur la donnée primaire validée (`load_final`, sources officielles House + Sénat),
**révisé à l'audit du 2026-07-03** : corrections read-time visibles, montants unifiés, ticker canonique,
enrichissements point-in-time, flags de traçabilité.

**Philosophie : on ne retire que l'AVÉRÉ inutilisable pour un backtest actions/ETF ; tout le reste est
GARDÉ et FLAGUÉ** (le doute se décide en aval, pas ici, et jamais en silence).

| Étape | Retire | Justification |
|---|---|---|
| A | dates illisibles / divulgation avant transaction / année implausible | chronologie inutilisable |
| B | sans ticker exploitable, non coté, famille non-action (bond/muni/gov/option) | pas de prix de marché |
| C | opérations hors achat/vente (échanges) | pas d'économie directionnelle claire |
| D | montant absent | sizing impossible |

⚠ Cette table contient des **lots réels** de lignes identiques (achats multi-comptes Self/Spouse/Joint —
colonnes `owner`/`occurrence_index`/`lot_size`) : **ne JAMAIS `drop_duplicates()`** dessus.


In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# --- localisation de la racine du dépôt (dossier contenant common/ ET data/) ---
# Le code+données S1/S2 ont été regroupés sous « 00_S1S2_donnees/ ». On teste le cwd et ses parents
# (cas normal : ce notebook est ouvert depuis 00_S1S2_donnees/), puis des candidats explicites au cas où
# Jupyter démarrerait à la RACINE du dépôt — 00_S1S2_donnees/ en est alors un ENFANT, jamais atteint par
# la simple remontée vers les parents.
def _find_repo():
    here = Path.cwd()
    cands = [here, *here.parents,
             here / "00_S1S2_donnees",
             Path.home() / "Downloads" / "Jupiter" / "00_S1S2_donnees",
             Path.home() / "Downloads" / "Jupiter"]            # repli : ancienne disposition (data/ à la racine)
    for c in cands:
        if (c / "common" / "quality.py").exists() and (c / "data" / "house" / "tables").exists():
            return c
    raise RuntimeError("Racine du dépôt (contenant common/ et data/) introuvable")

REPO = _find_repo()
sys.path.insert(0, str(REPO))

# --- fonctions réutilisées (aucune logique de nettoyage réécrite ici) ---
from common.quality import load_final, _asset_bucket    # chargement + famille d'actif
from common.schema import canonical_ticker               # ticker canonique (audit : ne confond pas
                                                          # le fonds coté NAN et l'artefact pandas 'nan')
from common.quiver_diagnosis import _quiver_untradeable  # ticker non coté (CUSIP, $, fragment OCR…)

# --- référentiels transverses (audit 2026-07-03) ---
REF = REPO / "data" / "reference"
renames = pd.read_csv(REF / "ticker_renames.csv")
sector_map = pd.read_csv(REF / "ticker_sector_map.csv").set_index("ticker")
print("Dépôt :", REPO)
print(f"Référentiels : {len(renames)} renommages/délistages, {len(sector_map)} tickers en carte secteur")

# --- petit utilitaire pour tracer l'entonnoir (étape, filtre, retirées, restantes) ---
funnel = []
def _step(code, label, before, after):
    funnel.append({"étape": code, "filtre": label,
                   "retirées": before - after, "restantes": after})
    print(f"[{code}] {label}\n    {before:,} → {after:,}   (retirées : {before - after:,})")

Dépôt : /Users/lemairealice/Downloads/Jupiter/00_S1S2_donnees
Référentiels : 84 renommages/délistages, 4848 tickers en carte secteur


## Chargement — notre donnée validée 2014-2026

`load_final` assemble les 4 sous-corpus (House / Sénat × électronique / OCR), déduplique les re-divulgations
d'une année sur l'autre, applique les corrections *read-time* (dates, tickers) et dérive les colonnes utiles
(`lag_days`, `op`, `amount_midpoint`, `corpus`…). On n'a donc rien à recalculer en amont.

In [2]:
df = load_final(REPO)
N0 = len(df)
print(f"{N0:,} transactions uniques chargées\n")
print("Répartition par sous-corpus :")
print(df["corpus"].value_counts().to_string())

# Réparations appliquées À LA LECTURE par load_final (le figé reste intact) — audit 2026-07-03 :
print(f"\nRéparations read-time déjà appliquées :")
print(f"  fourchettes tronquées à la borne basse réparées : {int(df['amount_range_repaired'].sum()):,}"
      f"  (montants ×2 corrigés, piste digitale House)")
print(f"  tickers récupérés depuis la description (double preuve) : "
      f"{int((df['ticker_source'] == 'recovered').sum()):,}")
_fix_bios = df['bioguide_id'].isin(['C001119', 'V000128', 'U000039', 'C001070'])
print(f"  identités réparées (Craig, Van Hollen, Udall, Casey) : {int(_fix_bios.sum())} lignes")


158,172 transactions uniques chargées

Répartition par sous-corpus :
corpus
House OCR             87011
House électronique    54150
Sénat électronique    13026
Sénat OCR              3985

Réparations read-time déjà appliquées :
  fourchettes tronquées à la borne basse réparées : 7,993  (montants ×2 corrigés, piste digitale House)
  tickers récupérés depuis la description (double preuve) : 2,319
  identités réparées (Craig, Van Hollen, Udall, Casey) : 53 lignes


## Normalisation des montants — palier ouvert au plancher + **midpoint unifié**

Deux conventions coexistaient dans le corpus (audit AMT-02) : `$1,001-$15,000` → 8 000,0 côté OCR House
mais 8 000,5 côté digital/Sénat. Ici, **une seule règle** : `amount_midpoint` = (borne basse + borne
haute) / 2 **recalculé depuis `amount_range`** quand la fourchette est complète ; un montant exact
(ex. `$584`) reste tel quel ; le palier ouvert « Over $50,000,000 » reste au plancher 50 M$.

In [3]:
# Palier ouvert « > $50M » : plancher cohérent inter-corpus (House OCR mettait 75M ; les autres, 50M).
palier_ouvert = df["amount_range"].astype(str).str.strip().eq("Over $50,000,000")
n_fix = int((palier_ouvert & (df["amount_midpoint"] != 50_000_000)).sum())
df.loc[palier_ouvert, "amount_midpoint"] = 50_000_000.0
print(f"Palier ouvert « > $50M » harmonisé au plancher : {n_fix} ligne(s).")

# Midpoint UNIFIÉ = (lo+hi)/2 exact depuis amount_range (fourchettes complètes uniquement).
import re as _re
def _mid_exact(a):
    nums = [int(x.replace(",", "")) for x in _re.findall(r"\$([\d,]+)", str(a))]
    return (nums[0] + nums[1]) / 2 if len(nums) == 2 else None
_mid = df["amount_range"].map(_mid_exact)
_maj = _mid.notna() & (df["amount_midpoint"] != _mid) & ~palier_ouvert
df.loc[_maj, "amount_midpoint"] = _mid[_maj]
print(f"Midpoint recalculé (lo+hi)/2 : {int(_maj.sum()):,} lignes alignées "
      f"(ancienne double convention .0 OCR / .5 digital-Sénat).")

Palier ouvert « > $50M » harmonisé au plancher : 2 ligne(s).
Midpoint recalculé (lo+hi)/2 : 85,416 lignes alignées (ancienne double convention .0 OCR / .5 digital-Sénat).


## Étape A — Dates présentes et cohérentes

Un backtest a besoin d'une **chronologie fiable**. On retire les lignes où :
- une date (transaction *ou* divulgation) est **illisible / absente** → `lag_days` vaut `NaN` (on ne sait pas
  *quand* agir) ;
- la **divulgation précède la transaction** (`lag_days < 0`) : c'est **impossible** (on « saurait » avant que
  le trade existe) — ce sont des coquilles du déposant ou des erreurs d'OCR ;
- garde-fou : l'**année de transaction** est hors plage plausible (avant 2012 ou après l'année de dépôt).

`lag_days` (= divulgation − transaction, en jours) est **déjà calculé** par `load_final`.

In [4]:
n = len(df)
fy = pd.to_numeric(df["file_year"], errors="coerce")
mask_parse    = df["lag_days"].notna()                        # dates lisibles
mask_coherent = df["lag_days"] >= 0                           # divulgation ≥ transaction
mask_year     = (df["txn_year"] >= 2012) & (df["txn_year"] <= fy)   # année plausible
keep = mask_parse & mask_coherent & mask_year

# aperçu : quelques dates incohérentes retirées (divulgation AVANT transaction)
apercu = (df[mask_parse & (df["lag_days"] < 0)]
          [["declarant_name", "ticker", "transaction_date", "disclosure_date", "lag_days"]].head(5))
print("Exemples de dates incohérentes retirées (divulgation avant transaction) :")
print(apercu.to_string(index=False), "\n")

df = df[keep].copy()
_step("A", "dates présentes & cohérentes", n, len(df))

Exemples de dates incohérentes retirées (divulgation avant transaction) :
  declarant_name ticker transaction_date disclosure_date  lag_days
     Kevin Yoder   SMLP       2014-12-17      2014-12-16      -1.0
Randy Neugebauer    QRE       2014-06-24      2014-06-20      -4.0
Randy Neugebauer    NaN       2014-05-15      2014-05-14      -1.0
  Nick J. Rahall     FB       2014-03-20      2014-03-12      -8.0
Patrick J Toomey    PFS       2014-11-30      2014-11-05     -25.0 

[A] dates présentes & cohérentes
    158,172 → 157,615   (retirées : 557)


## Étape B — Actions cotées et ETF uniquement

Un backtest a besoin d'un **prix**, donc d'un **ticker coté valide**. On garde une ligne si :
- son ticker se **normalise en un symbole non vide** (`norm_ticker`), **et**
- ce ticker est **réellement coté** (on écarte via `_quiver_untradeable` les CUSIP, préférentielles `$`,
  fragments d'OCR, échéances obligataires…), **et**
- sa **famille d'actif n'est pas explicitement non-cotée** (`_asset_bucket` ∉ {obligation, muni, gouvernement,
  option, « autre »}).

Concrètement, un **ticker « valide »** = un symbole qu'on peut relier à un **prix de marché** : **≤ 5 caractères**, lettres/chiffres, rien de bizarre.

| Ticker | Gardé ? | Raison |
|---|---|---|
| `AAPL`, `XOM`, `REGL` | ✅ | vrai symbole → on a un prix |
| *(vide)*, `NaN` | ❌ | aucun symbole |
| `BXS$A` | ❌ | le `$` = action préférentielle |
| `PFE  VTRS` | ❌ | espace = deux tickers collés (ligne d'échange) |
| CUSIP `037833100` | ❌ | numéro comptable, pas un ticker |

**On raisonne « ticker d'abord ».** Une ligne au ticker coté valide est gardée *même si son `asset_type` est
vide* — cas fréquent en House OCR : des milliers de vraies actions (NVDA, IBM, ASML…) sans étiquette de type.
Un filtre « type d'abord » les jetterait à tort (≈ 3 000 lignes).

En pratique, sur les 30 207 retirées : **~91 %** n'ont **aucun ticker** (non valorisables), **12 %** sont des
**options / obligations**, **< 1 %** des tickers malformés. La cellule d'audit ci-dessous le **prouve**,
chiffres à l'appui — rien n'est retiré « en aveugle ».

In [5]:
n = len(df)
NON_COTE = {"bond", "muni", "gov", "option", "autre"}
mask_ticker   = df["ticker"].map(lambda t: canonical_ticker(t)[0]) != ""   # symbole non vide (règle NAN corrigée)
mask_tradable = ~df["ticker"].map(_quiver_untradeable)                 # réellement coté
mask_famille  = ~df["asset_type"].map(_asset_bucket).isin(NON_COTE)    # pas une famille non-cotée
keep = mask_ticker & mask_tradable & mask_famille

apercu = df[~keep][["declarant_name", "asset_description", "asset_type", "ticker"]].head(5)
print("Exemples de lignes retirées (non cotées / non tickérisées) :")
print(apercu.to_string(index=False), "\n")

avant_B = df                       # snapshot avant filtrage (réutilisé par la cellule d'audit ci-dessous)
df = df[keep].copy()
_step("B", "actions + ETF cotés (ticker-first)", n, len(df))

Exemples de lignes retirées (non cotées / non tickérisées) :
  declarant_name                       asset_description asset_type ticker
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
    Lois Frankel Liberty broadband Corporation - Class C        NaN    NaN 

[B] actions + ETF cotés (ticker-first)
    157,615 → 129,102   (retirées : 28,513)


### Pourquoi ces lignes partent — contrôle chiffré

On réutilise l'état AVANT filtrage et les 3 masques de l'étape B (aucun re-filtrage) pour ventiler les
retraits par cause — chaque ligne écartée l'est pour une raison AVÉRÉE, jamais pour un simple champ vide.

In [6]:
# On réutilise `avant_B` (état AVANT filtrage) + les 3 masques calculés à l'étape B. On ne re-filtre RIEN.
fam = avant_B["asset_type"].map(_asset_bucket)
cause = pd.Series(index=avant_B.index, dtype="object")
cause[~mask_ticker]                                = "1. ticker VIDE (aucun symbole → pas de prix)"
cause[mask_ticker & ~mask_tradable]                = "2. ticker MALFORMÉ ($, espace, fragment OCR)"
cause[mask_ticker & mask_tradable & ~mask_famille] = "3. OPTION / OBLIGATION (famille non-cotée)"
ret = cause.dropna()

print(f"Pourquoi les {len(ret):,} lignes de l'étape B partent (une seule cause par ligne) :")
print(ret.value_counts().sort_index().to_string())
print()

print(f"Contrôle d'étanchéité — familles des {int(keep.sum()):,} GARDÉES (equity/ETF seulement) :")
print(fam[keep].value_counts().to_string())
fuite = int(fam[keep].isin(list(NON_COTE)).sum())
print(f"  → non-cotées ayant fui dans les gardées : {fuite}   (0 = filtre étanche)")
print()

rescue = int((keep & (fam == "manquant")).sum())
print(f"Rescue « ticker-first » : {rescue:,} actions/ETF à asset_type vide GARDÉES grâce au ticker.")

Pourquoi les 28,513 lignes de l'étape B partent (une seule cause par ligne) :
1. ticker VIDE (aucun symbole → pas de prix)    25759
2. ticker MALFORMÉ ($, espace, fragment OCR)      163
3. OPTION / OBLIGATION (famille non-cotée)       2591

Contrôle d'étanchéité — familles des 129,102 GARDÉES (equity/ETF seulement) :
asset_type
action      110019
manquant     17463
fonds         1620
  → non-cotées ayant fui dans les gardées : 0   (0 = filtre étanche)

Rescue « ticker-first » : 17,463 actions/ETF à asset_type vide GARDÉES grâce au ticker.


## Étape C — Direction exploitable (achat / vente)

Pour simuler une position, il faut un **sens clair**. On garde les **achats** et les **ventes** ; on retire les
`exchange` et les rares `other`, dont l'économie est ambiguë. *On ne présume pas la stratégie* : conserver les
deux sens laisse le backtest décider (long / short, étude d'événement, sorties). Le sens est déjà normalisé
dans la colonne `op` par `load_final`.

In [7]:
n = len(df)
df = df[df["op"].isin(["buy", "sell"])].copy()
_step("C", "direction ∈ {achat, vente}", n, len(df))

[C] direction ∈ {achat, vente}
    129,102 → 128,333   (retirées : 769)


## Étape D — Montant présent

Pour **dimensionner** une position (en notionnel), il faut un montant. On retire les lignes sans
`amount_midpoint` numérique (le point milieu de la fourchette déclarée, déjà calculé par `load_final`).
Coût négligeable (< 1 %).

In [8]:
n = len(df)
df = df[df["amount_midpoint"].notna()].copy()
_step("D", "montant présent (amount_midpoint)", n, len(df))

[D] montant présent (amount_midpoint)
    128,333 → 127,646   (retirées : 687)


## Enrichissements point-in-time et flags — **aucune ligne retirée ici**

Tout ce qui suit AJOUTE de l'information (colonnes) sans jamais filtrer :
1. **Parti point-in-time** à la date de la transaction (gère les switchs en cours de mandat via
   `party_affiliations` — Amash 2019, Mitchell 2018, Manchin 2024 ; l'ancienne colonne = parti du
   dernier mandat, 24 lignes fausses, audit PARTY-02) ;
2. **Commissions point-in-time** par Congrès (snapshots `data/reference/committees_snapshots/113..119` ;
   l'ancienne colonne = photo 2026 appliquée à toutes les années, audit COM-01) — bascule de Congrès
   au **3 janvier** des années impaires (audit CONG-01) ;
3. **Ticker canonique + renommages/délistages** (`ticker_yahoo` pour le join prix, référentiel
   `ticker_renames.csv` ; le champ `ticker` reste FIDÈLE à la déclaration) ;
4. **Classe d'actif / secteur / ETF proxy** (`ticker_sector_map.csv` : ETF diversifiés = `etf_broad`,
   PAS de faux secteur GICS) ;
5. **Flags de traçabilité** : dépôt tardif, lots multi-comptes, réparations.

In [9]:
# 1) Parti point-in-time — depuis les YAML congress-legislators versionnés (offline),
#    en éclatant party_affiliations (sous-périodes de switch EN COURS de mandat).
import yaml
try:
    from yaml import CSafeLoader as _YL
except ImportError:
    from yaml import SafeLoader as _YL

_ppl = []
for f in ("legislators-current.yaml", "legislators-historical.yaml"):
    _ppl += yaml.load((REPO / "data" / "house" / "reference" / f).read_text(), Loader=_YL)

PARTY_SPANS = {}
for p in _ppl:
    bio = (p.get("id") or {}).get("bioguide")
    if not bio:
        continue
    spans = []
    for t in (p.get("terms") or []):
        st, en = t.get("start"), t.get("end") or "2100-01-01"
        if not st:
            continue
        affs = t.get("party_affiliations")
        if affs:
            for a in affs:
                spans.append((pd.Timestamp(a.get("start") or st),
                              pd.Timestamp(a.get("end") or en), a.get("party")))
        else:
            spans.append((pd.Timestamp(st), pd.Timestamp(en), t.get("party")))
    if spans:
        PARTY_SPANS[bio] = sorted(spans)

def party_at(bio, d):
    spans = PARTY_SPANS.get(bio)
    if not spans or pd.isna(d):
        return None
    for st, en, pty in spans:
        if st <= d <= en:
            return pty
    before = [s for s in spans if s[0] <= d]
    return (before[-1] if before else spans[0])[2]

_old = df["party"].copy()
df["party"] = [party_at(b, d) or old for b, d, old in zip(df["bioguide_id"], df["_td"], _old)]
_chg = int((df["party"] != _old).sum())
print(f"Parti point-in-time : {_chg} ligne(s) corrigée(s) vs « parti du dernier mandat »")
print(df.loc[df["party"] != _old, ["declarant_name", "transaction_date", "party"]]
        .assign(ancien=_old[df["party"] != _old]).head(8).to_string(index=False))

Parti point-in-time : 24 ligne(s) corrigée(s) vs « parti du dernier mandat »
     declarant_name transaction_date      party      ancien
       Justin Amash       2015-11-10 Republican Libertarian
Joseph Manchin, III       2017-04-21   Democrat Independent
      Paul Mitchell       2018-12-17 Republican Independent
      Paul Mitchell       2018-12-17 Republican Independent
      Paul Mitchell       2018-12-12 Republican Independent
      Paul Mitchell       2018-12-12 Republican Independent
       Justin Amash       2018-11-01 Republican Libertarian
       Justin Amash       2018-11-01 Republican Libertarian


In [10]:
# 2) Commissions point-in-time — snapshots par Congrès (data/reference/committees_snapshots).
from collections import defaultdict

def congress_of(d):
    # Un Congrès commence le 3 JANVIER des années impaires : les trades des 1-2 janvier d'une année
    # impaire appartiennent encore au Congrès sortant (audit CONG-01).
    if pd.isna(d):
        return None
    y = d.year
    start_congress_year = y if y % 2 == 1 else y - 1
    if y % 2 == 1 and (d.month, d.day) < (1, 3):
        start_congress_year -= 2
    return 113 + (start_congress_year - 2013) // 2

SNAPS = {}
for cg in range(113, 120):
    d = REPO / "data" / "reference" / "committees_snapshots" / str(cg)
    mem = yaml.load((d / "membership.yaml").read_text(), Loader=_YL)
    com = yaml.load((d / "committees.yaml").read_text(), Loader=_YL)
    code_to_name = {c["thomas_id"]: c["name"] for c in com if "thomas_id" in c}
    bio2c = defaultdict(set)
    for code_, members in mem.items():
        cname = code_to_name.get(code_, code_)
        for m in members:
            if m.get("bioguide"):
                bio2c[m["bioguide"]].add(cname)
    SNAPS[cg] = {b: "; ".join(sorted(cs)) for b, cs in bio2c.items()}
    print(f"  Congrès {cg} : {len(SNAPS[cg])} membres avec commissions")

# Commissions « clés » : patterns LARGES documentés (fiscalité + défense + renseignement + banque,
# les deux chambres). La liste COMPLÈTE des commissions est exportée : la recherche peut redéfinir
# son propre flag sans re-générer la table.
KEY_PATTERNS = ("Financial Services", "Committee on Finance", "Ways and Means",
                "Banking", "Armed Services", "Intelligence")

df["congress"] = [congress_of(d) for d in df["_td"]]
_snap_get = lambda b, cg: (SNAPS.get(cg) or {}).get(b) if pd.notna(cg) and cg in SNAPS else None
df["committee_membership"] = [_snap_get(b, cg) for b, cg in zip(df["bioguide_id"], df["congress"])]
df["committees_key_flag"] = [any(p in m for p in KEY_PATTERNS) if isinstance(m, str) else pd.NA
                             for m in df["committee_membership"]]
print(f"\ncommissions PIT résolues : {df['committee_membership'].notna().mean():.1%} des lignes "
      f"| flag clé : {df['committees_key_flag'].mean():.1%}")

  Congrès 113 : 531 membres avec commissions
  Congrès 114 : 536 membres avec commissions


  Congrès 115 : 528 membres avec commissions
  Congrès 116 : 529 membres avec commissions


  Congrès 117 : 526 membres avec commissions
  Congrès 118 : 529 membres avec commissions


  Congrès 119 : 528 membres avec commissions

commissions PIT résolues : 99.1% des lignes | flag clé : 55.6%


In [11]:
# 3) Ticker canonique + renommages/délistages — `ticker` reste FIDÈLE à la déclaration.
_canon = df["ticker"].map(canonical_ticker)
df["ticker_yahoo"] = [c[0] for c in _canon]
df["flag_ticker"] = [c[1] for c in _canon]

_ren = renames.set_index("ticker_ancien")
_map_new = _ren.loc[(_ren["ticker_nouveau"].notna()) & (_ren["ticker_nouveau"] != ""), "ticker_nouveau"]
_n_ren = int(df["ticker_yahoo"].isin(_map_new.index).sum())
df["ticker_yahoo"] = df["ticker_yahoo"].map(lambda t: _map_new.get(t, t))

df["delist_type"] = df["ticker_yahoo"].map(_ren["type"]).where(
    df["ticker_yahoo"].isin(_ren.index[_ren["ticker_nouveau"].isna() | (_ren["ticker_nouveau"] == "")]))
# jambe absorbée d'une fusion : prix du successeur INVALIDE avant la fusion → prudence au join prix
_caution = set(_ren.index[(_ren.get("historique_valide") == "post_fusion_seulement")]) | \
           set(_ren.index[_ren["type"] == "recyclage_attention"])
_orig_canon = pd.Series([c[0] for c in _canon], index=df.index)
df["flag_price_caution"] = _orig_canon.isin(_caution) | df["ticker_yahoo"].isin(_caution)
df["is_delisted"] = df["delist_type"].notna()

print(f"ticker_yahoo : {_n_ren:,} lignes re-symbolisées (renommages/fusions), "
      f"{int(df['is_delisted'].sum()):,} lignes sur titres délistés (type le plus fréquent : "
      f"{df['delist_type'].mode().iat[0] if df['delist_type'].notna().any() else '—'}), "
      f"{int(df['flag_price_caution'].sum()):,} lignes « prudence join prix » (recyclage / jambe absorbée)")
print("flag_ticker :", df["flag_ticker"].value_counts().to_dict())

ticker_yahoo : 2,654 lignes re-symbolisées (renommages/fusions), 3,363 lignes sur titres délistés (type le plus fréquent : rachat_delisting), 472 lignes « prudence join prix » (recyclage / jambe absorbée)
flag_ticker : {'ok': 126848, 'classe_convertie': 775, 'contient_chiffre': 23}


In [12]:
# 4) Classe d'actif / secteur GICS / ETF proxy — carte transverse corrigée (audit SEC-ETF-01 :
#    les ETF diversifiés n'ont PAS de secteur ; l'ancienne colonne leur donnait un faux secteur LLM).
_key = pd.Series([c[0] for c in _canon], index=df.index)          # symbole déclaré canonisé (pré-rename)
df["asset_class"] = _key.map(sector_map["asset_class"]).fillna("unknown")
_sec_new = _key.map(sector_map["sector_gics"])
_etf_new = _key.map(sector_map["etf_proxy"])
# priorité à la carte quand elle connaît le ticker ; sinon on GARDE la colonne des FINAL (jamais de trou créé)
_known = _key.isin(sector_map.index)
df["sector_gics"] = _sec_new.where(_known & _sec_new.notna() & (_sec_new != ""), df["sector_gics"])
df.loc[_known & df["asset_class"].isin(["etf_broad", "etf_sector"]), "sector_gics"] = pd.NA
df["etf_proxy"] = _etf_new.where(_known & _etf_new.notna() & (_etf_new != ""), df["etf_proxy"])
df["is_broad_etf"] = df["asset_class"] == "etf_broad"

_stk = df["asset_class"].eq("stock")
print(f"asset_class : {df['asset_class'].value_counts().to_dict()}")
print(f"secteur GICS rempli (actions) : {df.loc[_stk, 'sector_gics'].notna().mean():.1%} "
      f"| etf_proxy rempli : {df['etf_proxy'].notna().mean():.1%}")

asset_class : {'stock': 122842, 'unknown': 3034, 'etf_broad': 1650, 'etf_sector': 120}
secteur GICS rempli (actions) : 100.0% | etf_proxy rempli : 98.8%


In [13]:
# 5) Flags de traçabilité — dépôts tardifs (l'info reste RÉELLE et exploitable à disclosure_date,
#    on FLAGUE, on ne retire pas — audit LAG-01/LAG-02) + lots multi-comptes (audit DUP-07).
df["flag_late_filing"] = df["lag_days"] > 45
df["flag_very_late_filing"] = df["lag_days"] > 365
df["lot_size"] = df.groupby("natural_key_hash")["natural_key_hash"].transform("size")
print(f"dépôts > 45 j : {int(df['flag_late_filing'].sum()):,} ({df['flag_late_filing'].mean():.1%}) "
      f"| > 365 j : {int(df['flag_very_late_filing'].sum()):,}")
print(f"lignes appartenant à un lot multi-lignes (mêmes 7 champs de clé) : "
      f"{int((df['lot_size'] > 1).sum()):,} — comptes multiples réels (owner/occurrence), PAS des doublons")

dépôts > 45 j : 15,083 (11.8%) | > 365 j : 2,891
lignes appartenant à un lot multi-lignes (mêmes 7 champs de clé) : 10,087 — comptes multiples réels (owner/occurrence), PAS des doublons


## Récapitulatif de l'entonnoir

In [14]:
rows = [{"étape": "—", "filtre": "départ (load_final)", "retirées": 0, "restantes": N0}] + funnel
recap = pd.DataFrame(rows)
print(recap.to_string(index=False))
print(f"\nDonnée propre finale : {len(df):,} lignes  ({100 * len(df) / N0:.1f} % du panel de départ)\n")

print("Par chambre :");     print(df["chamber"].value_counts().to_string())
print("\nPar sous-corpus :"); print(df["corpus"].value_counts().to_string())
print("\nPar sens :");        print(df["op"].value_counts().to_string())

étape                             filtre  retirées  restantes
    —                départ (load_final)         0     158172
    A       dates présentes & cohérentes       557     157615
    B actions + ETF cotés (ticker-first)     28513     129102
    C         direction ∈ {achat, vente}       769     128333
    D  montant présent (amount_midpoint)       687     127646

Donnée propre finale : 127,646 lignes  (80.7 % du panel de départ)

Par chambre :
chamber
house     115996
senate     11650

Par sous-corpus :
corpus
House OCR             72316
House électronique    43680
Sénat électronique     9729
Sénat OCR              1921

Par sens :
op
buy     65714
sell    61932


## Export — `data/clean/transactions_backtest_2014_2026.csv` (table de recherche canonique)

| Colonne | Sens |
|---|---|
| `bioguide_id`, `member_name`, `chamber`, `state_district` | identité du déposant (bioguide TOUJOURS rempli) |
| `party` | parti **à la date de la transaction** (point-in-time, switchs gérés) |
| `committee_membership`, `committees_key_flag`, `congress` | commissions **du Congrès de la transaction** (snapshots 113-119) ; flag = fiscalité/défense/renseignement/banque |
| `owner`, `occurrence_index`, `lot_size` | compte (Self/Spouse/Joint/Child), n° d'occurrence dans le dépôt, taille du lot de lignes identiques — **ne jamais `drop_duplicates()`** |
| `ticker` | symbole FIDÈLE à la déclaration |
| `ticker_yahoo`, `flag_ticker` | symbole canonique pour le join prix (format Yahoo + renommages appliqués) et statut de normalisation |
| `is_delisted`, `delist_type`, `flag_price_caution` | délistage (rachat/faillite) et prudence prix (recyclage de symbole, jambe absorbée de fusion) |
| `asset_class`, `asset_type`, `is_broad_etf` | stock / etf_sector / etf_broad / unknown ; type déclaré ; ETF diversifié |
| `sector_gics`, `etf_proxy` | secteur GICS (actions seulement — un ETF diversifié n'en a pas) et SPDR proxy |
| `direction` | buy / sell |
| `amount_midpoint`, `amount_range`, `amount_range_repaired` | milieu EXACT de fourchette (convention unique), fourchette source, flag de réparation de troncature |
| `transaction_date`, `disclosure_date`, `lag_days`, `flag_late_filing`, `flag_very_late_filing` | dates + retard de dépôt (>45 j légal, >365 j) |
| `doc_id`, `provenance`, `ticker_source`, `natural_key_hash` | traçabilité document/pipeline/récupération |
| `asset_description` | libellé source de l'actif |


In [15]:
OUT_DIR = REPO / "data" / "clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "transactions_backtest_2014_2026.csv"

COLS = ["bioguide_id", "member_name", "party", "chamber", "state_district",
        "committee_membership", "committees_key_flag", "congress",
        "owner", "occurrence_index", "lot_size",
        "ticker", "ticker_yahoo", "flag_ticker", "is_delisted", "delist_type", "flag_price_caution",
        "asset_class", "asset_type", "is_broad_etf", "sector_gics", "etf_proxy",
        "direction", "amount_midpoint", "amount_range", "amount_range_repaired",
        "transaction_date", "disclosure_date", "lag_days", "flag_late_filing", "flag_very_late_filing",
        "doc_id", "provenance", "ticker_source", "natural_key_hash", "asset_description"]

export = (df.rename(columns={"declarant_name": "member_name", "op": "direction"})
            .reindex(columns=COLS))

# Contrôles finaux — la table est CERTIFIÉE sur ces invariants :
assert export["bioguide_id"].notna().all() and (export["bioguide_id"] != "").all(), "bioguide manquant"
assert export["ticker"].notna().all(), "ticker manquant"
assert export["amount_midpoint"].notna().all(), "montant manquant"
assert export["direction"].isin(["buy", "sell"]).all(), "direction invalide"
assert (export["lag_days"] >= 0).all(), "chronologie incohérente"
assert export["natural_key_hash"].notna().all(), "hash manquant"

export.to_csv(OUT, index=False)
print(f"Écrit : {OUT}")
print(f"{len(export):,} lignes × {export.shape[1]} colonnes — tous les invariants vérifiés")
print(f"\nRépartition par chambre × ère :")
_y = pd.to_datetime(export["transaction_date"], errors="coerce").dt.year
print(pd.crosstab(export["chamber"], _y < 2020).rename(columns={True: "2014-2019", False: "2020-2026"}).to_string())

Écrit : /Users/lemairealice/Downloads/Jupiter/00_S1S2_donnees/data/clean/transactions_backtest_2014_2026.csv
127,646 lignes × 36 colonnes — tous les invariants vérifiés

Répartition par chambre × ère :
transaction_date  2020-2026  2014-2019
chamber                               
house                 66039      49957
senate                 4539       7111
